# Td-Tp 5 : Réseau de Neurones Récurrent - Partie I

Bienvenue dans le premier devoir du Cours 5 ! Dans ce devoir, vous allez implémenter votre premier Réseau de Neurones Récurrent en numpy.

# RNN classique
Les Réseaux de Neurones Récurrents (RNN) sont très efficaces pour le traitement du langage naturel et d'autres tâches séquentielles car ils ont de la "mémoire". Ils peuvent lire des entrées $x_t$ (comme des mots) une à la fois, et se souvenir de certaines informations/contexte à travers les activations de la couche cachée qui sont transmises d'un pas de temps au suivant. Cela permet à un RNN unidirectionnel de prendre en compte les informations du passé pour traiter les entrées ultérieures. Un RNN bidirectionnel peut prendre en compte le contexte à la fois du passé et du futur.

On utilise les notations suivantes :

* L'indice $i$ désigne un objet associé à la $i^{ème}$ couche. Exemple : $\sigma_{4}$ est la fonction d'activation de la $4^{ème}$ couche. 

* La lettre $x$ désigne une donnée d'entrée. Exemple : $x_{3}$ est l'entrée du $3^{ème}$ exemple d'entraînement.

* La lettre $h$ désigne l'état de la couche cachée. Exemple : $h_3$ est l'état actuel de la $3ème$ couche cachée.

* La lettre $t$ désigne le temps courant, et $T$ désigne le temps final.

* La lettre $y$ est réservée aux sorties : à l'instant $t$, $y_t$ désigne la sortie objective, et $\hat{y}_t$ désigne la sortie réelle.

* La lettre $l$ est réservée aux fonctions *loss*. $l$ pour la *loss* de chaque cellule, et $L$ pour la *loss* globale.

## Table des matières
- [1 - Propagation avant pour le Réseau de Neurones Récurrent de base](#1)
    - [1.1 - Cellule RNN](#1.1)
        - [Exercice 1 : rnn_cell_forward](#exo-1)
    - [1.2 - Propagation avant du RNN](#1.2)
        - [Exercice 2 : rnn_forward](#exo-2)
- [2 - Rétropropagation dans les réseaux de neurones récurrents](#2)
    - [2.1 - Rétropropagation du RNN de base](#2.1)
        - [Exercice 3 : Calcul des fonctions de rétropropagation d'une étape](#exo-3)
        - [Exercice 4 : rnn_cell_backward](#exo-4)
        - [Exercice 5 : rnn_backward](#exo-5)
- [3 - Réseau de Mémoire à Long Terme (LSTM)](#3)
    - [3.1 Cellule LSTM](#3.1)
        - [Exercice 6 : lstm_cell_forward](#exo-6)
    - [3.2 Passage avant pour LSTM](#3.2)
        - [Exercice 7 : lstm_forward](#exo-7)        
- [4 - Rétropropagation LSTM](#4)
    - [4.1 Un Pas en Arrière](#4.1)
        - [Exercice 8 : lstm_cell_backward](#exo-8) 
    - [4.2 Passage en arrière pour LSTM](#4.2)
        - [Exercice 9 : lstm_backward](#exo-9) 

In [1]:
import numpy as np
from rnn_utils import *

<a name='1'></a>
## 1 - Propagation avant pour le Réseau de Neurones Récurrent de base

Le RNN de base que vous allez implémenter a la structure ci-dessous. Dans cet exemple, $T_x = T_y$.

<img src="RNN.png" alt="image_description" width="800">

<caption><center> Figure 1 : Modèle basique de RNN</center></caption>

Voici comment vous pouvez implémenter un RNN :

**Étapes** :

1. Implémenter les calculs nécessaires pour un pas de temps du RNN.
2. Implémenter une boucle sur $T_x$ pas de temps afin de traiter toutes les entrées, une à la fois.

<a name='1.1'></a>
### 1.1 - Cellule RNN

Un réseau de neurones récurrent peut être vu comme la répétition d'une seule cellule. On va d'abord implémenter les calculs pour un seul pas de temps. La figure suivante décrit les opérations pour un pas de temps d'une cellule RNN.

<img src="RNN_cell.png" alt="image_description" width="400">
<caption><center> Figure 2 : Cellule RNN de base. Prend en entrée $x_i$ (entrée actuelle) et $h_{i-1}$ (état caché précédent contenant des informations du passé), et produit $h_i$ qui est donné à la cellule RNN suivante et aussi utilisé pour prédire $\hat{y}_i$. $u$, $v$ et $w$ représente les poids.</center></caption>

<a name='exo-1'></a>
**Exercice 1** : Implémenter la cellule RNN décrite dans la Figure (2).

**Instructions** :

1. Calculer l'état caché avec l'activation $\sigma_1 = \tanh$.
2. En utilisant votre nouvel état caché $h_i$, calculer la prédiction $\hat{y}_i$. On utilisera la fonction $\sigma_2 = softmax$.
3. Stocker $(h_i, h_{i-1}, x_i, parameters)$ dans le cache.
4. Retourner $h_i$, $\hat{y}_i$ et le cache.

Nous allons vectoriser sur $m$ exemples. Ainsi, $x_i$ aura la dimension $(d_x, n)$, et $h_i$ aura la dimension $(n_a, m)$.

In [2]:
def rnn_cell_forward(xt, h_prev, parameters):
    """
    Implémenter une seule étape de propagation avant de la cellule RNN comme décrit dans la Figure (2)

    Paramètres :
    xt -- données d'entrée au pas de temps "t", tableau numpy de forme (d_x, n).
    h_prev -- État caché au pas de temps "t-1", tableau numpy de forme (d_h, n)
    parameters -- dictionnaire Python contenant :
    w -- Matrice de poids multipliant l'entrée, tableau numpy de forme (d_x, d_h)
    v -- Matrice de poids multipliant l'état caché, tableau numpy de forme (d_h, d_h)
    y -- Matrice de poids reliant l'état caché à la sortie, tableau numpy de forme (d_h, d_y)
    bh -- Biais, tableau numpy de forme (d_h, 1)
    by -- Biais reliant l'état caché à la sortie, tableau numpy de forme (d_y, 1)
    
    La fonction retourne :
    h_next -- prochain état caché, de forme (d_a, b)
    yt_pred -- prédiction au pas de temps "t", tableau numpy de forme (d_y, n)
    cache -- tuple de valeurs nécessaires pour la passe arrière, contient (h_next, h_prev, xt, parameters)
    """
    
    ## Retrieve parameters from "parameters"
    w = parameters["w"]
    v = parameters["v"]
    u = parameters["u"]
    bh = parameters["bh"]
    by = parameters["by"]
    
    ## calculer le prochain état d'activation en utilisant la formule donnée ci-dessus
    ## np.tanh()
    # h_next = ...
    ########## 1 ligne ##########
    h_next = np.tanh(np.dot(w.T, xt) + np.dot(v.T, h_prev) + bh)
    #############################
    
    ## calculer la sortie de la cellule actuelle en utilisant la formule donnée ci-dessus
    ## softmax()
    # yt_pred = ...
    ########## 1 ligne ##########
    yt_pred = softmax(np.dot(u.T, h_next) + by)
    #############################
    
    ## stocker les valeurs dont vous avez besoin pour la propagation vers l'arrière dans le cache
    cache = (h_next, h_prev, xt, parameters)
    
    return h_next, yt_pred, cache

In [3]:
np.random.seed(1)
xt = np.random.randn(3,10)
h_prev = np.random.randn(5,10)
w = np.random.randn(3,5)
v = np.random.randn(5,5)
u = np.random.randn(5,2)
bh = np.random.randn(5,1)
by = np.random.randn(2,1)
parameters = {"w": w, "v": v, "u": u, "bh": bh, "by": by}

h_next, yt_pred, cache = rnn_cell_forward(xt, h_prev, parameters)
print("h_next[4] = ", h_next[4])
print("h_next.shape = ", h_next.shape)
print("yt_pred[1] =", yt_pred[1])
print("yt_pred.shape = ", yt_pred.shape)

h_next[4] =  [ 0.82818081 -0.24892282  0.84887426 -0.95095178  0.9835388  -0.87193051
  0.19919535 -0.96260404  0.99076946  0.8666213 ]
h_next.shape =  (5, 10)
yt_pred[1] = [0.95667287 0.49372474 0.5824185  0.01979334 0.43909214 0.83217006
 0.78442787 0.03778373 0.99695559 0.1020975 ]
yt_pred.shape =  (2, 10)


**Output attendu**: 

<table>
    <tr>
        <td>
            h_next[4]:
        </td>
        <td>
           [ 0.82818081 -0.24892282  0.84887426 -0.95095178  0.9835388  -0.87193051
  0.19919535 -0.96260404  0.99076946  0.8666213 ]
        </td>
    </tr>
        <tr>
        <td>
            h_next.shape:
        </td>
        <td>
           (5, 10)
        </td>
    </tr>
        <tr>
        <td>
            yt_pred[1]:
        </td>
        <td>
           [0.95667287 0.49372474 0.5824185  0.01979334 0.43909214 0.83217006
 0.78442787 0.03778373 0.99695559 0.1020975 ]
        </td>
    </tr>
        <tr>
        <td>
            yt_pred.shape:
        </td>
        <td>
           (2, 10)
        </td>
    </tr>

</table>

<a name='1.2'></a>
### 1.2 - Propagation avant du RNN

Un RNN peut être vu comme la répétition de la cellule qu'on vient de construire. Si la séquence de données d'entrée s'étend sur 10 pas de temps, alors on va copier la cellule RNN 10 fois. Chaque cellule prend en entrée l'état caché de la cellule précédente ($h_{t-1}$) et les données d'entrée du pas de temps actuel ($x_t$). Elle produit un état caché ($h_t$) et une prédiction ($\hat{y}_t$) pour ce pas de temps.

<img src="RNN_forward.png" alt="image_description" width="800">
<caption><center> Figure 3 : RNN de base. La séquence d'entrée $x = (x_1, x_2, ..., x_T)$ s'étend sur $T$ pas de temps. Le réseau produit $\hat{y} = (\hat{y}_1, \hat{y}_2, ..., \hat{y}_T)$. </center></caption>

<a name='exo-2'></a>
**Exercice 2** : Coder la propagation avant du RNN décrite dans la Figure (3).

**Instructions** :

1. Créer un vecteur de zéros ($h$) qui stockera tous les états cachés calculés par le RNN.
2. Initialiser l'état caché "suivant" comme $h_0$ (état caché initial).
3. Commencer à boucler sur chaque pas de temps, votre index incrémental est $t$ :
    - Mettre à jour l'état caché "suivant" et le cache en exécutant rnn_cell_forward,
    - Stocker l'état caché "suivant" dans $h$ (position $t$),
    - Stocker la prédiction dans $\hat{y}$,
    - Ajouter le cache à la liste des caches.
4. Retourner. $h$, $\hat{y}$ et caches.

In [4]:
def rnn_forward(x, h0, parameters):
    """
    Implémentez la propagation avant du réseau de neurones récurrent décrite dans la Figure (3).

    Paramètres :
    x -- Données d'entrée pour chaque pas de temps, de forme (d_x, n, T_x).
    h0 -- État caché initial, de forme (d_h, n)
    parameters -- dictionnaire Python contenant :
    w -- Matrice de poids multipliant l'entrée, tableau numpy de forme (d_x, d_h)
    v -- Matrice de poids multipliant l'état caché, tableau numpy de forme (d_h, d_h)
    u -- Matrice de poids reliant l'état caché à la sortie, tableau numpy de forme (d_h, d_y)
    bh -- Biais, tableau numpy de  forme (d_h, 1)
    by -- Biais reliant l'état caché à la sortie, tableau numpy de forme (d_y, 1)

    La fonction retourne :
    h -- États cachés pour chaque pas de temps, tableau numpy de forme (d_h, n, T_x)
    y_pred -- Prédictions pour chaque pas de temps, tableau numpy de forme (d_y, n, T_x)
    caches -- tuple de valeurs nécessaires pour la passe arrière, contenant (liste des caches, x)
    """
    
    ## Initialiser les "caches" qui contiendront la liste de tous les caches
    caches = []
    
    ## Récupérer les dimensions des formes de x et u
    d_x, n, T_x = x.shape
    d_h, d_y = parameters["u"].shape
    
    ## Initialiser "h" et "y_pred" avec des zéros
    # h = ...
    # y_pred = ...
    ######### 2 lignes ##########
    h = np.zeros((d_h, n, T_x))
    y_pred = np.zeros((d_y, n, T_x))
    #############################
    
    ## Initialiser h_next
    # h_next = ...
    ########## 1 ligne ##########
    h_next = h0
    #############################
    
    ## loop over all time-steps
    for t in range(T_x):
        ## Mettre à jour le prochain état caché, calculer la prédiction, récupérer le cache
        ## rnn_cell_forward
        ########## 1 ligne ##########
        h_next, yt_pred, cache = rnn_cell_forward(x[:,:,t], h_next, parameters)
        #############################
        ## Enregistrer la valeur du nouvel état caché dans "h"
        ########## 1 ligne ##########
        h[:,:,t] = h_next
        #############################
        ## Enregistrer la valeur de la prédiction dans "y_pred"
        ########## 1 ligne ##########
        y_pred[:,:,t] = yt_pred
        #############################
        ## Ajouter "cache" à "cache"
        ########## 1 ligne ##########
        caches.append(cache)
        #############################
    
    ## stocker les valeurs nécessaires à la propagation vers l'arrière dans le cache
    caches = (caches, x)
    
    return h, y_pred, caches

In [5]:
np.random.seed(1)
x = np.random.randn(3,10,4)
h0 = np.random.randn(5,10)
w = np.random.randn(3,5)
v = np.random.randn(5,5)
u = np.random.randn(5,2)
bh = np.random.randn(5,1)
by = np.random.randn(2,1)
parameters = {"w": w, "v": v, "u": u, "bh": bh, "by": by}

h, y_pred, caches = rnn_forward(x, h0, parameters)
print("h[4][1] = ", h[4][1])
print("h.shape = ", h.shape)
print("y_pred[1][3] =", y_pred[1][3])
print("y_pred.shape = ", y_pred.shape)
print("caches[1][1][3] =", caches[1][1][3])
print("len(caches) = ", len(caches))

h[4][1] =  [-0.35825902  0.59362885 -0.96347407 -0.75654416]
h.shape =  (5, 10, 4)
y_pred[1][3] = [0.05382839 0.0326001  0.04937623 0.02658246]
y_pred.shape =  (2, 10, 4)
caches[1][1][3] = [-1.1425182  -0.34934272 -0.20889423  0.58662319]
len(caches) =  2


**Output attendu** :

<table>
    <tr>
        <td>
            h[4][1] =
        </td>
        <td>
           [-0.35825902  0.59362885 -0.96347407 -0.75654416]
        </td>
    </tr>
        <tr>
        <td>
            h.shape =
        </td>
        <td>
           (5, 10, 4)
        </td>
    </tr>
        <tr>
        <td>
            y_pred[1][3] =
        </td>
        <td>
           [0.05382839 0.0326001  0.04937623 0.02658246]
        </td>
    </tr>
        <tr>
        <td>
            y_pred.shape =
        </td>
        <td>
           (2, 10, 4)
        </td>
    </tr>
        <tr>
        <td>
            caches[1][1][3] =
        </td>
        <td>
           [-1.1425182  -0.34934272 -0.20889423  0.58662319]
        </td>
    </tr>
        <tr>
        <td>
            len(caches) =
        </td>
        <td>
           2
        </td>
    </tr>

</table>

À cette étape, on a réussi à construire la propagation avant d'un réseau de neurones récurrent à partir de zéro. Cela fonctionnera suffisamment bien pour certaines applications, mais il souffre de problèmes de gradients qui disparaissent. Il fonctionne donc mieux lorsque chaque sortie $\hat{y}_t$ peut être estimée en utilisant principalement le contexte "local" (c'est-à-dire les informations provenant des entrées $x_{t'}$ où $t'$ n'est pas trop éloigné de $t$).

Une évolution possible de ce travail est de construire un modèle LSTM plus complexe, qui sera mieux adapté pour traiter les gradients qui disparaissent. Le LSTM sera mieux capable de se souvenir d'une information et de la conserver pendant plusieurs pas de temps.

<a name='2'></a>
## 2 - Rétropropagation dans les réseaux de neurones récurrents

Lorsque, dans un TD précédent, on a implémenté un réseau de neurones simple (entièrement connecté), on a utilisé la rétropropagation pour calculer les dérivées par rapport au coût afin de mettre à jour les paramètres. De même, dans les réseaux de neurones récurrents RNN, on peut calculer les dérivées par rapport au coût afin de mettre à jour les paramètres. Les équations de rétropropagation sont assez compliquées. On va les présenter brièvement ici :

<a name='2.1'></a>
### 2.1 - Rétropropagation du RNN de base

Nous allons commencer par calculer la rétropropagation pour la cellule RNN de base.

<img src="RNN_retro.png" style="width:500;height:300px;"> <br>

<caption><center> Figure 4: Rétropropagation de la cellule RNN. 
    
Tout comme dans un réseau de neurones entièrement connecté, la dérivée de la fonction de coût $L$ se propage à travers le RNN en suivant la règle de la chaîne du calcul différentiel. La règle de la chaîne est également utilisée pour calculer $(\frac{\partial L}{\partial w},\frac{\partial L}{\partial v},\frac{\partial L}{\partial b})$ (si le biais $b$ existe) afin de mettre à jour les paramètres $(w, v, b)$. </center></caption>

<a name='exo-3'></a>
**Exercice 3** Calcul des fonctions de rétropropagation d'une étape :

Pour calculer la fonction rnn_cell_backward, on doit calculer les équations suivantes : 
\begin{equation}
\begin{split}
\frac{\partial h_i}{\partial w} &= \sigma_1' (w^T.x_i + v^T. h_{i-1}).x_i \\
\frac{\partial h_i}{\partial v} &= \sigma_1' (w^T.x_i + v^T. h_{i-1}).h_{i-1} \\
\frac{\partial h_i}{\partial x_i} &= \sigma_1' (w^T.x_i + v^T. h_{i-1}).w \\
\frac{\partial L}{\partial h_{i-1}} &= \frac{\partial L}{\partial h_{i}} \frac{\partial h_i}{\partial h_{i-1}} \\
\frac{\partial h_i}{\partial h_{i-1}} &= \sigma_1' (w^T.x_i + v^T. h_{i-1}).v
\end{split}
\end{equation}

Pour une fonction d'activation $\sigma_1$ donnée, on calcule ces dérivées partielles en utilisant $\sigma_1'$. 

Dans un premier temps on choisira $\sigma_1 = \tanh$, dont la dérivée est $\sigma_1' = 1 - \tanh^2$. D'autres choix peuvent être considérés ultérieurement.

Il faudra également faire attention à la dimension des différents termes intervenant dans ces équations, afin de s'assurer que les produits matriciels ont du sens.

<a name='exo-4'></a>
**Exercice 4**

In [6]:
def rnn_cell_backward(dh_next, cache):
    """
    Implémenter la rétropropagation pour la cellule RNN (un seul pas de temps).

    Paramètres :
    dh_next -- Gradient de la perte par rapport à l'état caché suivant
    cache -- dictionnaire Python contenant des valeurs utiles (sortie de rnn_cell_forward())

    La fonction retorune :
    gradients -- dictionnaire Python contenant :
    dx -- Gradients des données d'entrée, de forme (d_x, n)
    dh_prev -- Gradients de l'état caché précédent, de forme (d_h, n)
    dw -- Gradients des poids de l'entrée vers caché, de forme (d_x, d_h)
    dv -- Gradients des poids cachés à cachés, de forme (d_h, d_h)
    dbh -- Gradients du vecteur de biais, de forme (d_h, 1)
    """
    
    ## Récupérer les valeurs du cache
    (h_next, h_prev, xt, parameters) = cache
    
    ## Récupérer les valeurs des paramètres
    w = parameters["w"]
    v = parameters["v"]
    u = parameters["u"]
    bh = parameters["bh"]
    by = parameters["by"]

    ## Calculer le gradient de tanh par rapport à h_next
    # dtanh = ...
    ########## 1 ligne ##########
    dtanh = (1 - h_next ** 2) * dh_next
    #############################

    ## Calculer le gradient de la perte par rapport à w
    # dxt = ...
    # dw = ...
    ########## 2 lignes #########
    dxt = np.dot(w, dtanh) 
    dw = np.dot(xt, dtanh.T)
    #############################

    ## Calculer le gradient par rapport à v
    # dh_prev = ...
    # dv = ...
    ########## 2 lignes #########
    dh_prev = np.dot(v, dtanh)
    dv = np.dot(h_prev, dtanh.T)
    #############################

    ## Calculer le gradient par rapport à bh
    # dbh = ...
    ########## 1 ligne ##########
    dbh = np.sum(dtanh, axis = 1,keepdims=1)
    #############################
    
    ## Stocker les gradients dans un dictionnaire Python
    gradients = {"dxt": dxt, "dh_prev": dh_prev, "dw": dw, "dv": dv, "dbh": dbh}
    
    return gradients

In [7]:
np.random.seed(1)
xt = np.random.randn(3,10)
h_prev = np.random.randn(5,10)
w = np.random.randn(3,5)
v = np.random.randn(5,5)
u = np.random.randn(5,2)
bh = np.random.randn(5,1)
by = np.random.randn(2,1)
parameters = {"w": w, "v": v, "u": u, "bh": bh, "by": by}

h_next, yt, cache = rnn_cell_forward(xt, h_prev, parameters)

dh_next = np.random.randn(5,10)
gradients = rnn_cell_backward(dh_next, cache)
print("gradients[\"dxt\"][1][2] =", gradients["dxt"][1][2])
print("gradients[\"dxt\"].shape =", gradients["dxt"].shape)
print("gradients[\"dh_prev\"][2][3] =", gradients["dh_prev"][2][3])
print("gradients[\"dh_prev\"].shape =", gradients["dh_prev"].shape)
print("gradients[\"dw\"][1][3] =", gradients["dw"][1][3])
print("gradients[\"dw\"].shape =", gradients["dw"].shape)
print("gradients[\"dv\"][1][2] =", gradients["dv"][1][2])
print("gradients[\"dv\"].shape =", gradients["dv"].shape)
print("gradients[\"dbh\"][4] =", gradients["dbh"][4])
print("gradients[\"dbh\"].shape =", gradients["dbh"].shape)

gradients["dxt"][1][2] = -0.6169635170803174
gradients["dxt"].shape = (3, 10)
gradients["dh_prev"][2][3] = -0.017219129729902586
gradients["dh_prev"].shape = (5, 10)
gradients["dw"][1][3] = 1.0728505835509434
gradients["dw"].shape = (3, 5)
gradients["dv"][1][2] = 0.40245182254944867
gradients["dv"].shape = (5, 5)
gradients["dbh"][4] = [1.82999491]
gradients["dbh"].shape = (5, 1)


**Output attendus**:

<table>
    <tr>
        <td>
            gradients["dxt"][1][2] =
        </td>
        <td>
            -0.6169635170803174
        </td>
    </tr>
        <tr>
        <td>
            gradients["dxt"].shape =
        </td>
        <td>
           (3, 10)
        </td>
    </tr>
        <tr>
        <td>
            gradients["dh_prev"][2][3] =
        </td>
        <td>
            -0.01721912972990269
        </td>
    </tr>
        <tr>
        <td>
            gradients["dh_prev"].shape =
        </td>
        <td>
           (5, 10)
        </td>
    </tr>
        <tr>
        <td>
            gradients["dw"][3][1] =
        </td>
        <td>
           1.0728505835509432
        </td>
    </tr>
            <tr>
        <td>
            gradients["dw"].shape =
        </td>
        <td>
           (3, 5)
        </td>
    </tr>
        <tr>
        <td>
            gradients["dv"][1][2] = 
        </td>
        <td>
            0.4024518225494487
        </td>
    </tr>
        <tr>
        <td>
            gradients["dv"].shape =
        </td>
        <td>
           (5, 5)
        </td>
    </tr>
        <tr>
        <td>
            gradients["dbh"][4] = 
        </td>
        <td>
            [1.82999491]
        </td>
    </tr>
        <tr>
        <td>
            gradients["dbh"].shape = 
        </td>
        <td>
           (5, 1)
        </td>
    </tr>
</table>

<a name='exo-5'></a>
**Exercice 5** Rétropropagation

Calculer les gradients du coût par rapport à $h_t$ à chaque pas de temps $t$ est utile car c'est ce qui aide le gradient à rétropropager vers la cellule RNN précédente. Pour ce faire, on doit parcourir tous les pas de temps en commençant par la fin, et à chaque étape, on incrémente les variables globales $db_h$, $dv$, $dw$ et on stocke $dx$.

**Instructions** :

Implémenter la fonction rnn_backward. Initialiser d'abord les variables de retour avec des zéros, puis parcourir tous les pas de temps tout en appelant rnn_cell_backward à chaque pas de temps, mettre à jour les autres variables en conséquence.

In [8]:
def rnn_backward(dh, caches):
    """
    Implémenter la rétropropagation pour un RNN sur l'ensemble d'une séquence de données d'entrée.

    Paramètres :
    dh -- Gradients amont de tous les états cachés, de forme (d_a, n, T_x)
    caches -- tuple contenant les informations de la passe avant (rnn_forward)

    La fonction retourne :
    gradients -- dictionnaire Python contenant :
        dx -- Gradient par rapport aux données d'entrée, tableau numpy de forme (d_x, n, T_x)
        dh0 -- Gradient par rapport à l'état caché initial, tableau numpy de forme (d_h, n)
        dw -- Gradient par rapport à la matrice de poids de l'entrée, tableau numpy de forme (d_x, d_h)
        dv -- Gradient par rapport à la matrice de poids de l'état caché, tableau numpy de forme (d_h, d_h)
        dbh -- Gradient par rapport au biais, de forme (d_h, 1)
    """
    
    ## Récupérer les valeurs du premier cache (t=1) des caches
    # ... = caches
    # ... = caches[0]
    ########## 2 lignes #########
    (caches, x) = caches
    (h1, h0, x1, parameters) = caches[0]
    #############################
    
    ## Récupérer les dimensions des formes dh et x1
    # ... = dh.shape
    # ... = x1.shape
    ########## 2 lignes #########
    d_h, n, T_x = dh.shape
    d_x, n = x1.shape
    #############################
    
    ## Initialiser les gradients avec les bonnes tailles `np.zeros(())`
    # dx = ...
    # dw = ...
    # dv = ...
    # dbh = ...
    # dh0 = ...
    # dh_prevt = ...
    ########## 6 lignes #########
    dx = np.zeros((d_x, n, T_x))
    dw = np.zeros((d_x, d_h))
    dv = np.zeros((d_h, d_h))
    dbh = np.zeros((d_h, 1))
    dh0 = np.zeros((d_h, n))
    dh_prevt = np.zeros((d_h, n))
    #############################
        
    
    ## Parcourer tous les pas de temps
    for t in reversed(range(T_x)):
        ## Calculez les gradients au pas de temps t. 
        ## Choisissez judicieusement le "dh_next" et le "cache" à utiliser dans l'étape de propagation vers l'arrière.
        # gradients = ...
        ########## 1 ligne ##########
        gradients = rnn_cell_backward(dh[:,:,t] + dh_prevt, caches[t])
        #############################
        ## Récupérer toutes les dérivées des gradients
        ######## 1~5 lignes #########
        dxt, dh_prevt, dwt, dvt, dbht = gradients["dxt"], gradients["dh_prev"], gradients["dw"], gradients["dv"], gradients["dbh"]
        #############################
        ## Incrémentez les dérivées globales par rapport aux paramètres en ajoutant leur dérivée au pas de temps t
        # dx[:, :, t] = ...
        # dw = ...
        # dv = ...
        # dbh = ...
        ########## 4 lignes #########
        dx[:, :, t] = dxt
        dw += dwt
        dv += dvt
        dbh += dbht
        #############################
        
    ## Définisser dh0 sur le gradient de a qui a été rétropropagé à travers tous les pas de temps
    # dh0 = ...
    ########## 1 ligne ##########
    dh0 = dh_prevt
    #############################

    ## Stocker les gradients dans un dictionnaire Python
    gradients = {"dx": dx, "dh0": dh0, "dw": dw, "dv": dv,"dbh": dbh}
    
    return gradients

In [9]:
np.random.seed(1)
x = np.random.randn(3,10,4)
h0 = np.random.randn(5,10)
w = np.random.randn(3,5)
v = np.random.randn(5,5)
u = np.random.randn(5,2)
bh = np.random.randn(5,1)
by = np.random.randn(2,1)
parameters = {"w": w, "v": v, "u": u, "bh": bh, "by": by}
h, y, caches = rnn_forward(x, h0, parameters)
dh = np.random.randn(5, 10, 4)
gradients = rnn_backward(dh, caches)

print("gradients[\"dx\"][1][2] =", gradients["dx"][1][2])
print("gradients[\"dx\"].shape =", gradients["dx"].shape)
print("gradients[\"dh0\"][2][3] =", gradients["dh0"][2][3])
print("gradients[\"dh0\"].shape =", gradients["dh0"].shape)
print("gradients[\"dw\"][1][3] =", gradients["dw"][1][3])
print("gradients[\"dw\"].shape =", gradients["dw"].shape)
print("gradients[\"dv\"][1][2] =", gradients["dv"][1][2])
print("gradients[\"dv\"].shape =", gradients["dv"].shape)
print("gradients[\"dbh\"][4] =", gradients["dbh"][4])
print("gradients[\"dbh\"].shape =", gradients["dbh"].shape)

gradients["dx"][1][2] = [-0.34598377 -0.00990713  0.11555775  0.01672709]
gradients["dx"].shape = (3, 10, 4)
gradients["dh0"][2][3] = -1.150045759605664
gradients["dh0"].shape = (5, 10)
gradients["dw"][1][3] = 7.513691445902343
gradients["dw"].shape = (3, 5)
gradients["dv"][1][2] = 0.48472650359440295
gradients["dv"].shape = (5, 5)
gradients["dbh"][4] = [-8.80702011]
gradients["dbh"].shape = (5, 1)


**Output attendus**:

<table>
    <tr>
        <td>
            gradients["dx"][1][2] =
        </td>
        <td>
           [-0.34598377 -0.00990713  0.11555775  0.01672709]
        </td>
    </tr>
        <tr>
        <td>
            gradients["dx"].shape =
        </td>
        <td>
           (3, 10, 4)
        </td>
    </tr>
        <tr>
        <td>
            gradients["dh0"][2][3] =
        </td>
        <td>
           -1.1500457596056641
        </td>
    </tr>
        <tr>
        <td>
            gradients["dh0"].shape =
        </td>
        <td>
           (5, 10)
        </td>
    </tr>
         <tr>
        <td>
            gradients["dw"][3][1] =
        </td>
        <td>
           7.513691445902342
        </td>
    </tr>
        <tr>
        <td>
            gradients["dw"].shape =
        </td>
        <td>
           (3, 5)
        </td>
    </tr>
        <tr>
        <td>
            gradients["dv"][1][2] = 
        </td>
        <td>
           0.4847265035944033
        </td>
    </tr>
        <tr>
        <td>
            gradients["dv"].shape =
        </td>
        <td>
           (5, 5)
        </td>
    </tr>
        <tr>
        <td>
            gradients["dbh"][4] = 
        </td>
        <td>
           [-8.80702011]
        </td>
    </tr>
        <tr>
        <td>
            gradients["dbh"].shape = 
        </td>
        <td>
           (5, 1)
        </td>
    </tr>
</table>

=========================================================
=========================================================
# LSTM


<a name='3'></a>
## 3 - Réseau de Mémoire à Long Terme (LSTM)

La figure suivante illustre le fonctionnement d'une cellule LSTM.

<img src="LSTM_cell.png" style="width:500;height:400px;">
<caption><center> Figure 1: Cellule LSTM. Celle-ci suit et met à jour un "état de cellule" ou une variable de mémoire $c_t$ à chaque pas de temps, qui peut être différente de $h_t$. </center></caption>

Similaire à l'exemple RNN ci-dessus, on commencera par implémenter la cellule LSTM pour un seul pas de temps. Ensuite, on pourra l'appeler itérativement dans une boucle `for` pour qu'elle traite une entrée avec $T_x$ pas de temps.

### À propos des portes

#### - Porte d'oubli

Pour illustrer cela, supposons qu'on lit des mots dans un texte et qu'on veut utiliser un LSTM pour suivre les structures grammaticales. On prend l'exemple d'un texte où on va vouloir observer l'évolution du sujet de chaque phrase entre les deux états : singulier ou pluriel. Si le sujet passe d'un mot singulier à un mot pluriel, on doit trouver un moyen de se débarrasser de la valeur de mémoire précédemment stockée de l'état singulier/pluriel. Dans un LSTM, la porte d'oubli nous permet de faire cela :

$$\begin{aligned}(\Gamma_f)_t &= \sigma_f(w_f^{\top}\cdot x_t + v_f^{\top} h_{t-1} + b_f) \\ &= \sigma_f(W_f\cdot [x_t, h_{t-1}] + b_f)\end{aligned}\tag{1} $$

Ici, $W_f = [w_f^{\top}, v_f^{\top}]$ représente les poids qui gouvernent le comportement de la porte d'oubli. On a fait le choix de concaténer $[x_t, h_{t-1}]$ et multiplier par $W_f$. L'équation ci-dessus donne un vecteur $(\Gamma_f)_t$ avec des valeurs entre 0 et 1. Ce vecteur de porte d'oubli sera multiplié élément par élément par l'état de cellule précédent $c_{t-1}$. Donc, si l'une des valeurs de $(\Gamma_f)_t$ est 0 (ou proche de 0), cela signifie que le LSTM doit supprimer cette information (par exemple, le sujet singulier) dans la composante correspondante de $c_{t-1}$. Si l'une des valeurs est 1, alors l'information sera conservée.

#### - Porte de mise à jour

Une fois qu'on a oublié que le sujet en discussion est singulier, on doit trouver un moyen de le mettre à jour pour refléter que le nouveau sujet est maintenant pluriel. Voici la formule pour la porte de mise à jour :

$$\begin{aligned}(\Gamma_i)_t &= \sigma_i(w_i^{\top}\cdot x_t + v_i^{\top} h_{t-1} + b_i) \\ &= \sigma_i(W_i\cdot [x_t, h_{t-1}] + b_i)\end{aligned}\tag{2} $$

Similaire à la porte d'oubli, ici $(\Gamma_i)_t$ est à nouveau un vecteur de valeurs entre 0 et 1. Cela sera multiplié élément par élément avec $\tilde{c}_t$, afin de calculer $c_t$.

#### - Mise à jour de la cellule

Pour mettre à jour le nouveau sujet, on doit créer un nouveau vecteur de nombres que nous pouvons ajouter à notre état de cellule précédent. L'équation qu'on va utiliser est :

$$ \begin{aligned}\tilde{c}_t &= \sigma_c(w_c^{\top}\cdot x_t + v_c^{\top} h_{t-1} + b_c) \\ &= \sigma_c(W_c\cdot [x_t, h_{t-1}] + b_c)\end{aligned}\tag{3} $$

Enfin, le nouvel état de la cellule est :

$$ c_t = (\Gamma_f)_t \odot c_{t-1} + (\Gamma_i)_t \odot \tilde{c}_t \tag{4} $$

#### - Porte de sortie

Pour décider quelles sorties on va considérer, on va utiliser les deux formules suivantes :

$$\begin{aligned}(\Gamma_o)_t &= \sigma_o(w_o^{\top}\cdot x_t + v_o^{\top} h_{t-1} + b_o) \\ &= \sigma_o(W_o\cdot [x_t, h_{t-1}] + b_o)\end{aligned}\tag{5}$$
$$ h_t = (\Gamma_o)_t \odot \sigma_{c,o} (c_t)\tag{6} $$

Où dans l'équation 5 on décide quoi sortir en utilisant une fonction sigmoïde et dans l'équation 6 on multiplie cela par la $\sigma_{c,o}$ (dans le schéma cette étape est décrite par "MàJ $c_t$") de l'état précédent.

#### - Remarque

Les fonctions d'activation $\sigma_f$, $\sigma_i$, $\sigma_c$, $\sigma_o$ et $\sigma_{c,o}$ sont à choisir par l'utilisateur. Dans un premier temps on peut les prendre toutes égales à $\tanh$.

<a name='3.1'></a>
### 3.1 - Cellule LSTM

<a name='exo-6'></a>
**Exercice 6** Implémentez la cellule LSTM décrite dans la Figure (1).

**Instructions**:
1. Concaténez $w$ et $v$ dans une matrice unique : $W = [w^{\top}, v^{\top}]$
2. Concaténez $x_t$ et $h_{t-1}$ dans une matrice unique : $concat = \begin{bmatrix} x_t \\ h_{t-1} \end{bmatrix}$
3. Calculez toutes les formules 1-6. Vous pouvez utiliser `sigmoid()` (fourni) et `np.tanh()`. $\sigma_f = sigmoid$, $\sigma_i = sigmoid$, $\sigma_c = \tanh$, $\sigma_o = sigmoid$, $\sigma_{c,o} = \tanh$.
4. Calculez la prédiction $y_t$. Vous pouvez utiliser `softmax()` (fourni).

In [10]:
## lstm_cell_forward

def lstm_cell_forward(xt, h_prev, c_prev, parameters):
    """
    Implémenter une seule étape vers l'avant de la cellule LSTM comme décrit dans la figure (1)

    Paramètres :
    xt - vos données d'entrée au pas de temps "t", tableau numpy de forme (d_x, n).
    h_prev -- État caché au pas de temps "t-1", tableau numpy de forme (d_h, n)
    c_prev -- État de la mémoire au pas de temps "t-1", tableau numpy de forme (d_h, n)
    parameters -- dictionnaire python contenant :
        Wf -- Matrice de poids de la porte oubliée, tableau numpy de forme (d_h, d_x + d_h)
           -- Wf = [wf.T vf.T], alors nous avons Wf*([xt h_prev].T)=wf.T*xt+vf.T*h_prev
        bf -- Biais de la porte oubliée, tableau numpy de forme (d_h, 1)
        Wi -- Matrice de poids de la porte d'entrée, tableau numpy de forme (d_h, d_x + d_h)
        bi - Biais de la porte d'entrée, tableau numpy de forme (d_h, 1)
        Wc -- Matrice de poids du premier "tanh", tableau numpy de forme (d_h, d_x + d_h)
        bc -- Biais du premier "tanh", tableau numpy de forme (d_h, 1)
        Wo -- Matrice de poids de la porte de sortie, tableau numpy de forme (d_h, d_x + d_h)
        bo -- Biais de la porte de sortie, tableau numpy de forme (d_h, 1)
        Wy -- Matrice de poids reliant l'état caché au tableau de forme numpy de sortie (d_y, d_h)
           -- Wy=wy.T
        by -- Biais reliant l'état caché au tableau de forme numpy de sortie (d_y, 1)

    Retour:
    h_next -- prochain état caché, de forme (d_h, n)
    c_next -- état de mémoire suivant, de forme (d_h, n)
    yt_pred -- prédiction au pas de temps "t", tableau numpy de forme (d_y, n)
    cache -- tuple de valeurs nécessaires pour la passe arrière, contient (h_next, c_next, h_prev, c_prev, xt, paramètres)

    Remarque : ft/it/ot représente les portes d'oubli/entrée/sortie, cct représente la valeur candidate (c tilde),
    c représente la valeur de la mémoire
    """

    ## Récupérer les paramètres de "parameters"
    Wf = parameters["Wf"]
    bf = parameters["bf"]
    Wi = parameters["Wi"]
    bi = parameters["bi"]
    Wc = parameters["Wc"]
    bc = parameters["bc"]
    Wo = parameters["Wo"]
    bo = parameters["bo"]
    Wy = parameters["Wy"]
    by = parameters["by"]
    
    ## Récupérer les dimensions des formes de xt et Wy
    d_x, n = xt.shape
    d_y, d_h = Wy.shape

    ## Concaténer h_prev et xt
    # concat = ...
    # ... = xt
    # ... = h_prev
    ######### 3 lignes ##########
    concat = np.zeros((d_x + d_h, n))
    concat[: d_x, :] = xt
    concat[d_x :, :] = h_prev
    #############################

    ## Calculer les valeurs pour ft, it, cct, c_next, ot, h_next en utilisant les formules données dans la figure (1)
    # ft = ...
    # it = ...
    # cct = ...
    # c_next = ...
    # ot = ...
    # h_next = ...
    ######### 6 lignes ##########
    ft = sigmoid(np.dot(Wf, concat) + bf)
    it = sigmoid(np.dot(Wi, concat) + bi)
    cct = np.tanh(np.dot(Wc, concat) + bc)
    c_next = ft * c_prev + it * cct
    ot = sigmoid(np.dot(Wo, concat) + bo)
    h_next = ot * np.tanh(c_next)
    #############################
    
    ## Calculer la prédiction de la cellule LSTM
    ########## 1 ligne ##########
    yt_pred = softmax(np.dot(Wy, h_next) + by)
    #############################

    ## stocker les valeurs nécessaires à la rétropropagation dans le cache
    cache = (h_next, c_next, h_prev, c_prev, ft, it, cct, ot, xt, parameters)

    return h_next, c_next, yt_pred, cache

In [11]:
np.random.seed(1)
xt = np.random.randn(3, 10)
h_prev = np.random.randn(5, 10)
c_prev = np.random.randn(5, 10)
Wf = np.random.randn(5, 3+5)
bf = np.random.randn(5, 1)
Wi = np.random.randn(5, 3+5)
bi = np.random.randn(5, 1)
Wo = np.random.randn(5, 3+5)
bo = np.random.randn(5, 1)
Wc = np.random.randn(5, 3+5)
bc = np.random.randn(5, 1)
Wy = np.random.randn(2, 5)
by = np.random.randn(2, 1)

parameters = {"Wf": Wf, "Wi": Wi, "Wo": Wo, "Wc": Wc, "Wy": Wy, "bf": bf, "bi": bi, "bo": bo, "bc": bc, "by": by}

h_next, c_next, yt, cache = lstm_cell_forward(xt, h_prev, c_prev, parameters)
print("h_next[4] = ", h_next[4])
print("h_next.shape = ", c_next.shape)
print("c_next[2] = ", c_next[2])
print("c_next.shape = ", c_next.shape)
print("yt[1] =", yt[1])
print("yt.shape = ", yt.shape)
print("cache[1][3] =", cache[1][3])
print("len(cache) = ", len(cache))

h_next[4] =  [-0.03449407  0.00894276  0.30298006  0.43189303 -0.66647818  0.49405344
  0.07131081  0.29639761 -0.17613571 -0.05742235]
h_next.shape =  (5, 10)
c_next[2] =  [-0.46575925  1.20869986  0.41886424  0.56972286 -1.47213921  0.01343717
  0.11796743 -0.8771188  -0.82891074 -0.54510368]
c_next.shape =  (5, 10)
yt[1] = [0.47525553 0.149091   0.09999298 0.07675625 0.85582416 0.06982482
 0.28651579 0.31146423 0.68286299 0.38685244]
yt.shape =  (2, 10)
cache[1][3] = [-1.28569784  0.47890445  0.97067993 -0.69978423  0.20420028 -0.62734419
 -0.33943755 -0.22803275  0.15418059  0.77555587]
len(cache) =  10


**Output attendus**:

<table>
    <tr>
        <td>
            h_next[4] =
        </td>
        <td>
           [-0.03449407  0.00894276  0.30298006  0.43189303 -0.66647818  0.49405344
  0.07131081  0.29639761 -0.17613571 -0.05742235]
        </td>
    </tr>
        <tr>
        <td>
            h_next.shape =
        </td>
        <td>
           (5, 10)
        </td>
    </tr>
        <tr>
        <td>
            c_next[2] =
        </td>
        <td>
           [-0.46575925  1.20869986  0.41886424  0.56972286 -1.47213921  0.01343717
  0.11796743 -0.8771188  -0.82891074 -0.54510368]
        </td>
    </tr>
        <tr>
        <td>
            c_next.shape =
        </td>
        <td>
           (5, 10)
        </td>
    </tr>
        <tr>
        <td>
            yt[1] = 
        </td>
        <td>
           [0.47525553 0.149091   0.09999298 0.07675625 0.85582416 0.06982482
 0.28651579 0.31146423 0.68286299 0.38685244]
        </td>
    </tr>
        <tr>
        <td>
            yt.shape =
        </td>
        <td>
           (2, 10)
        </td>
    </tr>
    <tr>
        <td>
            cache[1][3] =
        </td>
        <td>
           [-1.28569784  0.47890445  0.97067993 -0.69978423  0.20420028 -0.62734419
 -0.33943755 -0.22803275  0.15418059  0.77555587]
        </td>
    </tr>
        <tr>
        <td>
            len(cache) =
        </td>
        <td>
           10
        </td>
    </tr>
</table>

<a name='3.2'></a>
### 3.2 - Passage avant pour LSTM

Maintenant qu'on a implémenté un pas de LSTM, on peut maintenant itérer cela en utilisant une boucle `for` pour traiter une séquence de $T_x$ entrées.

<img src="LSTM_cells.png" style="width:500;height:300px;">
<caption><center> Figure 2: LSTM sur plusieurs pas de temps. </center></caption>

<a name='exo-7'></a>
**Exercice 7** Implémentez `lstm_forward()` pour faire fonctionner un LSTM sur $T_x$ pas de temps.

**Note**: $c_0$ est initialisé avec des zéros.

In [12]:
# lstm_forward

def lstm_forward(x, h0, parameters):
    """
    Implémentez la propagation vers l’avant du réseau neuronal récurrent à l’aide d’une cellule LSTM décrite dans la figure (2).
    
    Paramètres:
    x -- Données d'entrée pour chaque pas de temps, de forme (d_x, n, T_x).
    h0 -- État caché initial, de forme (d_h, n)
    parameters -- dictionnaire python contenant :
                Wf -- Matrice de poids de la porte d'oubli, tableau numpy de forme (d_h, d_x + d_h)
                bf -- Biais de la porte d'oubli, tableau numpy de forme (d_h, 1)
                Wi -- Matrice de poids de la porte d'entrée, tableau numpy de forme (d_h, d_x + d_h)Weight matrix of the input gate, numpy array of shape (d_h, d_x + d_h)
                bi -- Biais de la porte d'entrée, tableau numpy de forme (d_h, 1)
                Wc -- Matrice de poids du premier "tanh", tableau numpy de forme (d_h, d_x + d_h)
                bc -- Biais du premier "tanh", tableau numpy de forme (d_h, 1)
                Wo -- Matrice de poids de la porte de sortie, tableau numpy de forme (d_h, d_x + d_h)
                bo -- Biais de la porte de sortie, tableau numpy de forme (d_h, 1)
                Wy -- Matrice de poids reliant l'état caché à la sortie, tableau numpy de forme (d_y, d_h)
                by -- Biais reliant l'état caché à la sortie, tableau numpy de forme (d_y, 1)
                        
    Retours:
    h -- États cachés pour chaque pas de temps, tableau numpy de forme (d_h, n, T_x)
    y -- Prédictions pour chaque pas de temps, tableau numpy de forme (d_y, n, T_x)
    caches -- tuple de valeurs nécessaires à la passe arrière, contient (liste de tous les caches, x)
    """

    ## Initialiser les "caches", qui suivront la liste de tous les caches
    caches = []
    
    ## Récupérer les dimensions des formes de x et Wy
    # ... = x.shape
    # ... = parameters["Wy"].shape
    ######### 2 lignes ##########
    d_x, n, T_x = x.shape
    d_y, d_h = parameters["Wy"].shape
    #############################
    
    ## initialiser "h", "c" et "y" avec des zéros
    # h = ...
    # c = ...
    # y = ...
    ######### 3 lignes ##########
    h = np.zeros((d_h, n, T_x))
    c = np.zeros((d_h, n, T_x))
    y = np.zeros((d_y, n, T_x))
    #############################
    
    ## Initialisez h_next et c_next
    # h_next = ...
    # c_next = ...
    ######### 2 lignes ##########
    h_next = h0
    c_next = np.zeros(h_next.shape)
    #############################
    
    ## boucler sur tous les pas de temps
    for t in range(T_x):
        ## Mettre à jour le prochain état caché, le prochain état de la mémoire, calculer la prédiction, récupérer le cache
        ########## 1 ligne ##########
        h_next, c_next, yt, cache = lstm_cell_forward(x[:, :, t], h_next, c_next, parameters)
        #############################
        
        ## Enregistrez la valeur du nouvel état caché "h_next" dans h
        ########## 1 ligne ##########
        h[:,:,t] = h_next
        #############################
        
        ## Enregistrer la valeur de la prédiction en y
        ########## 1 ligne ##########
        y[:,:,t] = yt
        #############################
        
        ## Enregistrer la valeur de l'état de cellule suivant
        ########## 1 ligne ##########
        c[:,:,t] = c_next
        #############################
        
        ## Ajouter le cache dans les caches
        ########## 1 ligne ##########
        caches.append(cache)
        #############################
        
    
    ## stocker les valeurs nécessaires à la propagation vers l'arrière dans le cache
    caches = (caches, x)

    return h, y, c, caches

In [13]:
np.random.seed(1)
x = np.random.randn(3, 10, 7)
h0 = np.random.randn(5, 10)
Wf = np.random.randn(5, 3+5)
bf = np.random.randn(5, 1)
Wi = np.random.randn(5, 3+5)
bi = np.random.randn(5, 1)
Wo = np.random.randn(5, 3+5)
bo = np.random.randn(5, 1)
Wc = np.random.randn(5, 3+5)
bc = np.random.randn(5, 1)
Wy = np.random.randn(2, 5)
by = np.random.randn(2, 1)

parameters = {"Wf": Wf, "Wi": Wi, "Wo": Wo, "Wc": Wc, "Wy": Wy, "bf": bf, "bi": bi, "bo": bo, "bc": bc, "by": by}

h, y, c, caches = lstm_forward(x, h0, parameters)
print("h[4][3][6] = ", h[4][3][6])
print("h.shape = ", h.shape)
print("y[1][4][3] =", y[1][4][3])
print("y.shape = ", y.shape)
print("caches[1][1[1]] =", caches[1][1][1])
print("c[1][2][1] = ", c[1][2][1])
print("len(caches) = ", len(caches))

h[4][3][6] =  0.2677338226003625
h.shape =  (5, 10, 7)
y[1][4][3] = 0.6500921491842742
y.shape =  (2, 10, 7)
caches[1][1[1]] = [ 0.82797464  0.23009474  0.76201118 -0.22232814 -0.20075807  0.18656139
  0.41005165]
c[1][2][1] =  -0.6462156740716575
len(caches) =  2


**Output attendus**:

<table>
    <tr>
        <td>
           h[4][3][6] =
        </td>
        <td>
           0.26773382260036244
        </td>
    </tr>
    <tr>
        <td>
           h.shape =
        </td>
        <td>
           (5, 10, 7)
        </td>
    </tr>
    <tr>
        <td>
           y[1][4][3] =
        </td>
        <td>
           0.6500921491842743
        </td>
    </tr>
    <tr>
        <td>
           y.shape =
        </td>
        <td>
           (2, 10, 7)
        </td>
    </tr>
    <tr>
        <td>
           caches[1][1][1] =
        </td>
        <td>
           [ 0.82797464  0.23009474  0.76201118 -0.22232814 -0.20075807  0.18656139
  0.41005165]
        </td>
     </tr>
     <tr>
        <td>
           c[1][2][1] =
        </td>
        <td>
           -0.6462156740716574
        </td>
    </tr>       
    <tr>
        <td>
           len(caches) =
        </td>
        <td>
           2
        </td>
    </tr>
</table>

<a name='4'></a>
## 4 - Rétropropagation LSTM

<a name='4.1'></a>
### 4.1 Un Pas en Arrière
<img src="LSTM_cells.png" style="width:500;height:300px;">

La rétropropagation du LSTM est légèrement plus compliquée que la propagation avant. Dans un premier temps, il faut calculer les dérivées des équations des différentes portes. Les voici :

**Indication** : 
1. $c_{next}$ répresente $c_{t+1}$, $c_{prev}$ répresente $c_{t}$
2. $h_{next}$ répresente $h_{t+1}$, $h_{prev}$ répresente $h_{t}$
3. $dh_{next} = \frac{\partial L_2}{\partial h_{next}}$ et $dc_{next} = \frac{\partial L_2}{\partial c_{next}}$

**Dérivées des Portes**

$$d(\Gamma_o)_t = dh_{next} \odot \tanh(c_{next}) \odot (\Gamma_o)_t \odot (1-(\Gamma_o)_t) \tag{7}$$

$$d\tilde c_t = \left(dc_{next} + (\Gamma_o)_t (1-\tanh(c_{next})^2) \odot dh_{next}\right) \odot (\Gamma_i)_t \odot (1-(\tilde c_t)^2) \tag{8}$$

$$d(\Gamma_i)_t = \left(dc_{next} + (\Gamma_o)_t (1-\tanh(c_{next})^2) \odot dh_{next}\right) \odot \tilde c_t \odot (\Gamma_i)_t \odot (1-(\Gamma_i)_t) \tag{9}$$

$$d(\Gamma_f)_t = \left(dc_{next} + (\Gamma_o)_t (1-\tanh(c_{next})^2) \odot dh_{next}\right) \odot c_{prev} \odot (\Gamma_f)_t \odot (1-(\Gamma_f)_t) \tag{10}$$

**Dérivées des Paramètres**

$$ dW_f = d(\Gamma_f)_t \cdot \begin{pmatrix} x_t \\ h_{prev}\end{pmatrix}^{\top} \tag{11} $$
$$ dW_i = d(\Gamma_i)_t \cdot \begin{pmatrix} x_t \\ h_{prev}\end{pmatrix}^{\top} \tag{12} $$
$$ dW_c = d\tilde c_{next} \cdot \begin{pmatrix} x_t \\ h_{prev}\end{pmatrix}^{\top} \tag{13} $$
$$ dW_o = d(\Gamma_o)_t \cdot \begin{pmatrix} x_t \\ h_{prev}\end{pmatrix}^{\top} \tag{14}$$

Pour calculer $db_f, db_u, db_c, db_o$, il suffit de sommer sur l'axe horizontal (axis=1) sur $d(\Gamma_f)_t, d(\Gamma_i)_t, d\tilde c_t, d(\Gamma_o)_t$ respectivement. Noter qu'il faut avoir l'option `keep_dims = True`.

Enfin, on va calculer la dérivée par rapport à l'état caché précédent, à l'état de mémoire précédent et à l'entrée.

$$ dx_t = w_f^{\top}\cdot d(\Gamma_f)_t + w_i^{\top}\cdot d(\Gamma_i)_t + w_c^{\top}\cdot d\tilde c_t + w_o^{\top}\cdot d(\Gamma_o)_t \tag{15} $$
Ici, les poids $w^{\top} = W[:, :d_x]$.
$$ dh_{prev} = v_f^{\top}\cdot d(\Gamma_f)_t + v_i^{\top}\cdot d(\Gamma_i)_t + v_c^{\top}\cdot d\tilde c_t + v_o^{\top}\cdot d(\Gamma_o)_t \tag{16}$$
Ici, les poids $v^{\top} = W[:, d_x:]$.
$$ dc_{prev} = dc_{next} \cdot (\Gamma_f)_t + (\Gamma_o)_t \cdot (1- \tanh(c_{next})^2) \cdot (\Gamma_f)_t \cdot dh_{next} \tag{17}$$

<a name='exo-8'></a>
**Exercice 8** Implémenter `lstm_cell_backward` en programmant les équations $7-17$ ci-dessus.

In [14]:
def lstm_cell_backward(dh_next, dc_next, cache):
    """
    Implémenter le passage en arrière pour la cellule LSTM (pas de temps unique).

    Paramètres:
    dh_next -- Gradients du prochain état caché, de forme (d_h, n)
    dc_next -- Gradients de l'état de cellule suivant, de forme (d_h, n)
    cache -- cache stockant les informations de la passe avant

    Retours:
    gradients -- dictionnaire python contenant :
            dxt -- Gradient des données d'entrée au pas de temps t, de forme (d_x, n)
            dh_prev -- Gradient par rapport à l'état caché précédent, tableau numpy de forme (d_h, n)
            dc_prev -- Gradient par rapport à l'état mémoire précédent, de forme (d_h, n, T_x)
            dWf -- Gradient par rapport à la matrice de poids de la porte d'oubli, tableau numpy de forme (d_h, d_x + d_h)
            dWi -- Gradient par rapport à la matrice de poids de la porte d'entrée, tableau numpy de forme (d_h, d_x + d_h)
            dWc -- Gradient par rapport à la matrice de poids de la porte mémoire, tableau numpy de forme (d_h, d_x + d_h)
            dWo -- Gradient par rapport à la matrice de poids de la porte de sortie, tableau numpy de forme (d_h, d_x + d_h)
            dbf -- Gradient par rapport à biais de la porte d'oubli, de forme (d_h, 1)
            dbi -- Gradient par rapport à biais de la porte d'entrée, de forme (d_h, 1)
            dbc -- Gradient par rapport à biais de la porte mémoire, de forme (d_h, 1)
            dbo -- Gradient par rapport à biais de la porte de sortie, de forme (d_h, 1)
    """

    ## Récupérer des informations du "cache"
    (h_next, c_next, h_prev, c_prev, ft, it, cct, ot, xt, parameters) = cache
    
    ## Récupérer les dimensions de la forme xt et h_next
    ######### 2 lignes ##########
    d_x, n = xt.shape
    d_h, n = h_next.shape
    #############################
    
    ## Calculer les dérivés liés aux portes en examinant attentivement les équations (7) à (10)
    ######### 4 lignes ##########
    dot = dh_next * np.tanh(c_next) * ot * (1 - ot)
    dcct = (dh_next * ot * (1 - np.tanh(c_next) ** 2) + dc_next) * it * (1 - cct ** 2)
    dit = (dh_next * ot * (1 - np.tanh(c_next) ** 2) + dc_next) * cct * (1 - it) * it
    dft = (dh_next * ot * (1 - np.tanh(c_next) ** 2) + dc_next) * c_prev * ft * (1 - ft)
    #############################

    ## Calculer les dérivées liées aux paramètres. Utiliser les équations (11)-(14)
    # dWf, dWi, dWc, dWo, dbf, dbi, dbc, dbo
    ######### 8 lignes ##########
    dWf = np.dot(dft, np.hstack([xt.T,h_prev.T]))
    dWi = np.dot(dit, np.hstack([xt.T,h_prev.T]))
    dWc = np.dot(dcct, np.hstack([xt.T,h_prev.T]))
    dWo = np.dot(dot, np.hstack([xt.T,h_prev.T]))
    dbf = np.sum(dft, axis=1, keepdims=True)
    dbi = np.sum(dit, axis=1, keepdims=True)
    dbc = np.sum(dcct, axis=1, keepdims=True)
    dbo = np.sum(dot, axis=1, keepdims=True)
    #############################

    ## Calculer les dérivés par rapport à l'état caché précédent, à l'état de la mémoire précédent et à l'entrée. 
    ## Utilisez les équations (15) à (17).
    # dh_prev, dc_prev, dxt
    ######### 3 lignes ##########
    dxt = np.dot(Wf[:, :d_x].T, dft) + np.dot(Wc[:, :d_x].T, dcct) + np.dot(Wi[:, :d_x].T, dit) + np.dot(Wo[:, :d_x].T, dot)
    dh_prev = np.dot(Wf[:, d_x:].T, dft) + np.dot(Wc[:, d_x:].T, dcct) + np.dot(Wi[:, d_x:].T, dit) + np.dot(Wo[:, d_x:].T, dot)
    dc_prev = (dh_next * ot * (1 - np.tanh(c_next) ** 2) + dc_next) * ft
    #############################
    
    ## Enregistrer les gradients dans le dictionnaire
    gradients = {"dxt": dxt, "dh_prev": dh_prev, "dc_prev": dc_prev, "dWf": dWf,"dbf": dbf, "dWi": dWi,"dbi": dbi,
                "dWc": dWc,"dbc": dbc, "dWo": dWo,"dbo": dbo}

    return gradients

In [15]:
np.random.seed(1)
xt = np.random.randn(3,10)
h_prev = np.random.randn(5,10)
c_prev = np.random.randn(5,10)
Wf = np.random.randn(5, 5+3)
bf = np.random.randn(5,1)
Wi = np.random.randn(5, 5+3)
bi = np.random.randn(5,1)
Wo = np.random.randn(5, 5+3)
bo = np.random.randn(5,1)
Wc = np.random.randn(5, 5+3)
bc = np.random.randn(5,1)
Wy = np.random.randn(2,5)
by = np.random.randn(2,1)

parameters = {"Wf": Wf, "Wi": Wi, "Wo": Wo, "Wc": Wc, "Wy": Wy, "bf": bf, "bi": bi, "bo": bo, "bc": bc, "by": by}

h_next, c_next, yt, cache = lstm_cell_forward(xt, h_prev, c_prev, parameters)

dh_next = np.random.randn(5,10)
dc_next = np.random.randn(5,10)
gradients = lstm_cell_backward(dh_next, dc_next, cache)
print("gradients[\"dxt\"][1][2] =", gradients["dxt"][1][2])
print("gradients[\"dxt\"].shape =", gradients["dxt"].shape)
print("gradients[\"dh_prev\"][2][3] =", gradients["dh_prev"][2][3])
print("gradients[\"dh_prev\"].shape =", gradients["dh_prev"].shape)
print("gradients[\"dc_prev\"][2][3] =", gradients["dc_prev"][2][3])
print("gradients[\"dc_prev\"].shape =", gradients["dc_prev"].shape)
print("gradients[\"dWf\"][3][1] =", gradients["dWf"][3][1])
print("gradients[\"dWf\"].shape =", gradients["dWf"].shape)
print("gradients[\"dWi\"][1][2] =", gradients["dWi"][1][2])
print("gradients[\"dWi\"].shape =", gradients["dWi"].shape)
print("gradients[\"dWc\"][3][1] =", gradients["dWc"][3][1])
print("gradients[\"dWc\"].shape =", gradients["dWc"].shape)
print("gradients[\"dWo\"][1][2] =", gradients["dWo"][1][2])
print("gradients[\"dWo\"].shape =", gradients["dWo"].shape)
print("gradients[\"dbf\"][4] =", gradients["dbf"][4])
print("gradients[\"dbf\"].shape =", gradients["dbf"].shape)
print("gradients[\"dbi\"][4] =", gradients["dbi"][4])
print("gradients[\"dbi\"].shape =", gradients["dbi"].shape)
print("gradients[\"dbc\"][4] =", gradients["dbc"][4])
print("gradients[\"dbc\"].shape =", gradients["dbc"].shape)
print("gradients[\"dbo\"][4] =", gradients["dbo"][4])
print("gradients[\"dbo\"].shape =", gradients["dbo"].shape)

gradients["dxt"][1][2] = 2.4651199732572406
gradients["dxt"].shape = (3, 10)
gradients["dh_prev"][2][3] = 0.3186338415351005
gradients["dh_prev"].shape = (5, 10)
gradients["dc_prev"][2][3] = 0.7764145026413788
gradients["dc_prev"].shape = (5, 10)
gradients["dWf"][3][1] = -0.6419801969136802
gradients["dWf"].shape = (5, 8)
gradients["dWi"][1][2] = 0.16786190385221555
gradients["dWi"].shape = (5, 8)
gradients["dWc"][3][1] = -1.357014345112671
gradients["dWc"].shape = (5, 8)
gradients["dWo"][1][2] = 0.42858196806199683
gradients["dWo"].shape = (5, 8)
gradients["dbf"][4] = [-1.10640531]
gradients["dbf"].shape = (5, 1)
gradients["dbi"][4] = [0.22097232]
gradients["dbi"].shape = (5, 1)
gradients["dbc"][4] = [-2.09655781]
gradients["dbc"].shape = (5, 1)
gradients["dbo"][4] = [-0.058674]
gradients["dbo"].shape = (5, 1)


**Output attendus**:

<table>
    <tr>
        <td>
           gradients["dxt"][1][2] =
        </td>
        <td>
           2.4651199732572406
        </td>
    </tr>
        <tr>
        <td>
           gradients["dxt"].shape =
        </td>
        <td>
           (3, 10)
        </td>
    </tr>
        <tr>
        <td>
           gradients["dh_prev"][2][3] =
        </td>
        <td>
           0.31863384153510066
        </td>
    </tr>
        <tr>
        <td>
           gradients["dh_prev"].shape =
        </td>
        <td>
           (5, 10)
        </td>
    </tr>
         <tr>
        <td>
           gradients["dc_prev"][2][3] =
        </td>
        <td>
           0.7764145026413788
        </td>
    </tr>
        <tr>
        <td>
           gradients["dc_prev"].shape =
        </td>
        <td>
           (5, 10)
        </td>
    </tr>
        <tr>
        <td>
           gradients["dWf"][3][1] = 
        </td>
        <td>
           -0.6419801969136802
        </td>
    </tr>
        <tr>
        <td>
           gradients["dWf"].shape =
        </td>
        <td>
           (5, 8)
        </td>
    </tr>
        <tr>
        <td>
           gradients["dWi"][1][2] = 
        </td>
        <td>
           0.1678619038522156
        </td>
    </tr>
        <tr>
        <td>
           gradients["dWi"].shape = 
        </td>
        <td>
           (5, 8)
        </td>
    </tr>
    <tr>
        <td>
           gradients["dWc"][3][1] = 
        </td>
        <td>
           -1.357014345112671
        </td>
    </tr>
        <tr>
        <td>
           gradients["dWc"].shape = 
        </td>
        <td>
           (5, 8)
        </td>
    </tr>
    <tr>
        <td>
           gradients["dWo"][1][2] = 
        </td>
        <td>
           0.42858196806199683
        </td>
    </tr>
        <tr>
        <td>
           gradients["dWo"].shape = 
        </td>
        <td>
           (5, 8)
        </td>
    </tr>
    <tr>
        <td>
           gradients["dbf"][4] = 
        </td>
        <td>
           [-1.10640531]
        </td>
    </tr>
        <tr>
        <td>
           gradients["dbf"].shape = 
        </td>
        <td>
           (5, 1)
        </td>
    </tr>
    <tr>
        <td>
           gradients["dbi"][4] = 
        </td>
        <td>
           [0.22097232]
        </td>
    </tr>
        <tr>
        <td>
           gradients["dbi"].shape = 
        </td>
        <td>
           (5, 1)
        </td>
    </tr>
        <tr>
        <td>
           gradients["dbc"][4] = 
        </td>
        <td>
           [-2.09655781]
        </td>
    </tr>
        <tr>
        <td>
           gradients["dbc"].shape = 
        </td>
        <td>
           (5, 1)
        </td>
    </tr>
        <tr>
        <td>
           gradients["dbo"][4] = 
        </td>
        <td>
           [-0.058674]
        </td>
    </tr>
        <tr>
        <td>
           gradients["dbo"].shape = 
        </td>
        <td>
           (5, 1)
        </td>
    </tr>
</table>

<a name='4.2'></a>
### 4.2 Passage en arrière pour LSTM

Cette partie est très similaire à la fonction `rnn_backward` que vous avez implémentée la dernière fois. Vous allez d’abord créer des variables de même dimension que vos variables de retour. Vous parcourirez ensuite tous les pas de temps en commençant par la fin et appellerez la fonction en une étape que vous avez implémentée pour LSTM à chaque itération. Vous mettrez ensuite à jour les paramètres en les additionnant individuellement. Renvoyez enfin un dictionnaire avec les nouveaux dégradés.

<a name='exo-9'></a>
**Exercice 9** Implémentez la fonction `lstm_backward`.

**Instructions** : Créez une boucle for commençant à $T_x$ et remontant en arrière. Pour chaque étape, appelez `lstm_cell_backward` et mettez à jour vos anciens dégradés en leur ajoutant les nouveaux dégradés. Notez que `dxt` n'est pas mis à jour mais est stocké.

In [16]:
def lstm_backward(dh, caches):
    
    """
    Implémenter le passage arrière pour le RNN avec la cellule LSTM (sur une séquence entière).

    Paramètres:
    dh -- Gradients par rapport aux états cachés, tableau numpy de forme (d_h, n, T_x)
    dc -- Gradients par rapport aux états de mémoire, tableau numpy de forme (d_h, n, T_x)
    caches -- cache stockant les informations de la passe avant (lstm_forward)

    Retours:
    gradients -- dictionnaire python contenant :
                dx -- Gradient d'entrées, de forme (d_x, n, T_x)
                dh0 -- Gradient par rapport à l'état caché précédent, tableau numpy de forme (d_h, n)
                dWf -- Gradient par rapport à la matrice de poids de la porte d'oubli, tableau numpy de forme (d_h, d_x + d_h)
                dWi -- Gradient par rapport à la matrice de poids de la porte d'entrée, tableau numpy de forme (d_h, d_x + d_h)
                dWc -- Gradient par rapport à la matrice de poids de la porte mémoire, tableau numpy de forme (d_h, d_x + d_h)
                dWo -- Gradient par rapport à la matrice de poids de la porte de sortie, tableau numpy de forme (d_h, d_x + d_h)
                dbf -- Gradient par rapport à biais de la porte d'oubli, de forme (d_h, 1)
                dbi -- Gradient par rapport à biais de la porte d'entrée, de forme (d_h, 1)
                dbc -- Gradient par rapport à biais de la porte mémoire, de forme (d_h, 1)
                dbo -- Gradient par rapport à biais de la porte de sortie, de forme (d_h, 1)
    """

    ## Récupèrer les valeurs du premier cache (t=1) des caches.
    (caches, x) = caches
    (h1, c1, h0, c0, f1, i1, cc1, o1, x1, parameters) = caches[0]
    
    ## Récupérer les dimensions des formes dh et x1
    ######### 2 lignes ##########
    d_h, n, T_x = dh.shape
    d_x, n = x1.shape
    #############################
    
    ## initialiser les gradients avec les bonnes tailles
    # dx, dh0, dh_prevt, dc_prevt, dWf, dWi, dWc, dWo, dbf, dbi, dbc, dbo
    ######### 12 lignes ##########
    dx = np.zeros((d_x, n, T_x))
    dh0 = np.zeros((d_h, n))
    dh_prevt = np.zeros((d_h, n))
    dc_prevt = np.zeros((d_h, n))
    dWf = np.zeros((d_h, d_x + d_h))
    dWi = np.zeros((d_h, d_x + d_h))
    dWc = np.zeros((d_h, d_x + d_h))
    dWo = np.zeros((d_h, d_x + d_h))
    dbf = np.zeros((d_h, 1))
    dbi = np.zeros((d_h, 1))
    dbc = np.zeros((d_h, 1))
    dbo = np.zeros((d_h, 1))
    ##############################
    
    ## boucler sur toute la séquence
    for t in reversed(range(T_x)):
        ## Calculer tous les dégradés à l'aide de lstm_cell_backward
        ########## 1 ligne ##########
        gradients = lstm_cell_backward(dh[:,:,t] + dh_prevt, dc_prevt, caches[t])
        #############################
        ## Stocker ou ajouter le gradient au gradient de l'étape précédente des paramètres
        # dx, dWf, dWi, dWc, dWo, dbf, dbi, dbc, dbo
        ########## 9 lignes #########
        dx[:,:,t] = gradients["dxt"]
        dWf += gradients["dWf"]
        dWi += gradients["dWi"]
        dWc += gradients["dWc"]
        dWo += gradients["dWo"]
        dbf += gradients["dbf"]
        dbi += gradients["dbi"]
        dbc += gradients["dbc"]
        dbo += gradients["dbo"]
        #############################
    ## Définissez le gradient de la première activation sur le gradient rétropropagé dh_prev.
    ########## 1 ligne ##########
    dh0 = gradients["dh_prev"]
    #############################

    ## Stocker les gradients dans un dictionnaire python
    gradients = {"dx": dx, "dh0": dh0, "dWf": dWf,"dbf": dbf, "dWi": dWi,"dbi": dbi,
                "dWc": dWc,"dbc": dbc, "dWo": dWo,"dbo": dbo}
    
    return gradients

In [17]:
np.random.seed(1)
x = np.random.randn(3, 10, 7)
h0 = np.random.randn(5, 10)
Wf = np.random.randn(5, 3+5)
bf = np.random.randn(5, 1)
Wi = np.random.randn(5, 3+5)
bi = np.random.randn(5, 1)
Wo = np.random.randn(5, 3+5)
bo = np.random.randn(5, 1)
Wc = np.random.randn(5, 3+5)
bc = np.random.randn(5, 1)

parameters = {"Wf": Wf, "Wi": Wi, "Wo": Wo, "Wc": Wc, "Wy": Wy, "bf": bf, "bi": bi, "bo": bo, "bc": bc, "by": by}

h, y, c, caches = lstm_forward(x, h0, parameters)

dh = np.random.randn(5, 10, 4)
gradients = lstm_backward(dh, caches)

print("gradients[\"dx\"][1][2] =", gradients["dx"][1][2])
print("gradients[\"dx\"].shape =", gradients["dx"].shape)
print("gradients[\"dh0\"][2][3] =", gradients["dh0"][2][3])
print("gradients[\"dh0\"].shape =", gradients["dh0"].shape)
print("gradients[\"dWf\"][3][1] =", gradients["dWf"][3][1])
print("gradients[\"dWf\"].shape =", gradients["dWf"].shape)
print("gradients[\"dWi\"][1][2] =", gradients["dWi"][1][2])
print("gradients[\"dWi\"].shape =", gradients["dWi"].shape)
print("gradients[\"dWc\"][3][1] =", gradients["dWc"][3][1])
print("gradients[\"dWc\"].shape =", gradients["dWc"].shape)
print("gradients[\"dWo\"][1][2] =", gradients["dWo"][1][2])
print("gradients[\"dWo\"].shape =", gradients["dWo"].shape)
print("gradients[\"dbf\"][4] =", gradients["dbf"][4])
print("gradients[\"dbf\"].shape =", gradients["dbf"].shape)
print("gradients[\"dbi\"][4] =", gradients["dbi"][4])
print("gradients[\"dbi\"].shape =", gradients["dbi"].shape)
print("gradients[\"dbc\"][4] =", gradients["dbc"][4])
print("gradients[\"dbc\"].shape =", gradients["dbc"].shape)
print("gradients[\"dbo\"][4] =", gradients["dbo"][4])
print("gradients[\"dbo\"].shape =", gradients["dbo"].shape)

gradients["dx"][1][2] = [ 0.3310824   0.59249749  0.28928419 -0.60016464]
gradients["dx"].shape = (3, 10, 4)
gradients["dh0"][2][3] = 0.3052255922438375
gradients["dh0"].shape = (5, 10)
gradients["dWf"][3][1] = 0.6019703050240847
gradients["dWf"].shape = (5, 8)
gradients["dWi"][1][2] = 0.3746048574483192
gradients["dWi"].shape = (5, 8)
gradients["dWc"][3][1] = 2.121343196185227
gradients["dWc"].shape = (5, 8)
gradients["dWo"][1][2] = 0.28508004618134386
gradients["dWo"].shape = (5, 8)
gradients["dbf"][4] = [-0.20193342]
gradients["dbf"].shape = (5, 1)
gradients["dbi"][4] = [0.01166503]
gradients["dbi"].shape = (5, 1)
gradients["dbc"][4] = [0.06262086]
gradients["dbc"].shape = (5, 1)
gradients["dbo"][4] = [0.05065611]
gradients["dbo"].shape = (5, 1)


**Output attendus**:

<table>
    <tr>
        <td>
           gradients["dx"][1][2] =
        </td>
        <td>
           [ 0.3310824   0.59249749  0.28928419 -0.60016464]
        </td>
    </tr>
        <tr>
        <td>
           gradients["dx"].shape =
        </td>
        <td>
           (3, 10, 4)
        </td>
    </tr>
        <tr>
        <td>
           gradients["dh0"][2][3] =
        </td>
        <td>
           0.30522559224383766
        </td>
    </tr>
        <tr>
        <td>
           gradients["dh0"].shape =
        </td>
        <td>
           (5, 10)
        </td>
    </tr>
        <tr>
        <td>
           gradients["dWf"][3][1] = 
        </td>
        <td>
           0.6019703050240847
        </td>
    </tr>
        <tr>
        <td>
           gradients["dWf"].shape =
        </td>
        <td>
           (5, 8)
        </td>
    </tr>
        <tr>
        <td>
           gradients["dWi"][1][2] = 
        </td>
        <td>
           0.3746048574483191
        </td>
    </tr>
        <tr>
        <td>
           gradients["dWi"].shape = 
        </td>
        <td>
           (5, 8)
        </td>
    </tr>
    <tr>
        <td>
           gradients["dWc"][3][1] = 
        </td>
        <td>
           2.121343196185227
        </td>
    </tr>
        <tr>
        <td>
           gradients["dWc"].shape = 
        </td>
        <td>
           (5, 8)
        </td>
    </tr>
    <tr>
        <td>
           gradients["dWo"][1][2] = 
        </td>
        <td>
           0.2850800461813438
        </td>
    </tr>
        <tr>
        <td>
           gradients["dWo"].shape = 
        </td>
        <td>
           (5, 8)
        </td>
    </tr>
    <tr>
        <td>
           gradients["dbf"][4] = 
        </td>
        <td>
           [-0.20193342]
        </td>
    </tr>
        <tr>
        <td>
           gradients["dbf"].shape = 
        </td>
        <td>
           (5, 1)
        </td>
    </tr>
    <tr>
        <td>
           gradients["dbi"][4] = 
        </td>
        <td>
           [0.01166503]
        </td>
    </tr>
        <tr>
        <td>
           gradients["dbi"].shape = 
        </td>
        <td>
           (5, 1)
        </td>
    </tr>
        <tr>
        <td>
           gradients["dbc"][4] = 
        </td>
        <td>
           [0.06262086]
        </td>
    </tr>
        <tr>
        <td>
           gradients["dbc"].shape = 
        </td>
        <td>
           (5, 1)
        </td>
    </tr>
        <tr>
        <td>
           gradients["dbo"][4] = 
        </td>
        <td>
           [0.05065611]
        </td>
    </tr>
        <tr>
        <td>
           gradients["dbo"].shape = 
        </td>
        <td>
           (5, 1)
        </td>
    </tr>
</table>